In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
def create_dataset(n_samples: int = 1500, random_seed: int = 42) -> pd.DataFrame:
    np.random.seed(random_seed)
    sq_ft = np.random.normal(loc=2000, scale=500, size=n_samples).clip(600, 4500)
    bedrooms = np.random.choice([1, 2, 3, 4, 5], size=n_samples, p=[0.1, 0.25, 0.35, 0.2, 0.1])
    bathrooms = (bedrooms * 0.75 + np.random.normal(0, 0.3, size=n_samples)).round().clip(1, 5)
    walkability_score = np.random.uniform(20, 100, size=n_samples)
    house_age = np.random.exponential(scale=15, size=n_samples).clip(0, 70)x
    true_intercept = 120_000
    noise = np.random.normal(loc=0, scale=25_000, size=n_samples)
    price = (
        true_intercept
        + (140 * sq_ft)
        + (15_000 * bedrooms)
        + (18_000 * bathrooms)
        + (800 * walkability_score)
        - (1_200 * house_age)
        + noise
    )
    df = pd.DataFrame({
        "sq_ft": sq_ft.round(1),
        "bedrooms": bedrooms,
        "bathrooms": bathrooms,
        "walkability_score": walkability_score.round(1),
        "house_age": house_age.round(1),
        "price": price.round(2)
    })
    return df
if __name__ == "__main__":
    Path("data").mkdir(exist_ok=True)
    df = create_dataset()
    df.to_csv("data/housing_features.csv", index=False)
    print("Dataset generated successfully at data/housing_features.csv")

Dataset generated successfully at data/housing_features.csv


In [6]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def run_experiment():
    df = pd.read_csv("data/housing_features.csv")
    X = df[["sq_ft", "bedrooms", "bathrooms", "walkability_score", "house_age"]]
    y = df["price"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression())
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    print("--- MODEL PERFORMANCE METRICS ---")
    print(f"R-squared (R2) Score:        {r2:.4f}")
    print(f"Root Mean Squared Error:     ${rmse:,.2f}")
    print(f"Mean Absolute Error:         ${mae:,.2f}")
    Path("plots").mkdir(exist_ok=True)
    plt.figure(figsize=(7, 5))
    plt.scatter(y_test, y_pred, alpha=0.5, edgecolors='k', color='#1f77b4')
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
    plt.xlabel("Actual Price ($)")
    plt.ylabel("Predicted Price ($)")
    plt.title("Actual vs. Predicted Property Valuations")
    plt.tight_layout()
    plt.savefig("plots/actual_vs_predicted.png", dpi=300)
    plt.close()
if __name__ == "__main__":
    run_experiment()

--- MODEL PERFORMANCE METRICS ---
R-squared (R2) Score:        0.9133
Root Mean Squared Error:     $26,071.05
Mean Absolute Error:         $20,861.84


<ipython-input-6-3870cbfaba5f>:39: MatplotlibDeprecationWarning: The x parameter as float was deprecated in Matplotlib 3.10 and will be removed in 3.12. Use int(x) instead.
  plt.savefig("plots/actual_vs_predicted.png", dpi=300)
<ipython-input-6-3870cbfaba5f>:39: MatplotlibDeprecationWarning: The y parameter as float was deprecated in Matplotlib 3.10 and will be removed in 3.12. Use int(y) instead.
  plt.savefig("plots/actual_vs_predicted.png", dpi=300)
